# Day 10 — Solution: Diagnostics

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 7)
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

def ols_fit(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    s2 = e @ e / (n - 2)
    se = np.sqrt(np.diag(s2 * XtX_inv))
    return {"b": b, "se": se, "resid": e, "yhat": X @ b, "X": X, "XtX_inv": XtX_inv}

## E1 — the page

In [ ]:
from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    rr = np.log(get_prices(["AAPL", "SPY"], start="2015-01-01")).diff().dropna()
else:
    p = synthetic_prices(n_days=2500, n_assets=2, seed=23, corr=0.5)
    rr = np.log(p).diff().dropna(); rr.columns = ["AAPL", "SPY"]

f = ols_fit(rr["SPY"].values, rr["AAPL"].values)
x, e, yhat = rr["SPY"].values, f["resid"], f["yhat"]
idx = rr.index

fig, ax = plt.subplots(2, 2, figsize=(10, 7))
ax[0, 0].scatter(yhat, e, s=4, alpha=0.3); ax[0, 0].set_title("1. e vs fitted (A1 curve? A4 fan?)")
ax[0, 1].plot(idx, e, lw=0.4); ax[0, 1].set_title("2. e vs time (regimes/breaks?)")
stats.probplot(e, dist="norm", plot=ax[1, 0]); ax[1, 0].set_title("3. QQ vs normal (A6)")
ax[1, 1].scatter(x, np.abs(e), s=4, alpha=0.3); ax[1, 1].set_title("4. |e| vs x (variance loads on |x|?)")
plt.tight_layout(); plt.show()

**Demands (exemplar).** Panel 1: no curve; wide vertical band (idio vol) —
demands nothing structural. Panel 2: |e| clusters stack at 2020 (real) or
vol-regime bands (synthetic) → **demands White** (A4) — and notice the
clusters align with the day-3 stylized facts. Panel 3: heavy S-tails vs
normal → accept as A6-managed-by-n; optionally compare t(5) which
straightens — a tail-robustness note, not an SE change. Panel 4: mild fan,
|e| largest mid-|x| extremes → consistent with the White call; no
curvature claimed. **The page's value is the demands list it emits — the
figure without the list is decoration.**

## E2 — leverage and Cook's D

In [ ]:
X, XtX_inv = f["X"], f["XtX_inv"]
n, k = X.shape
h = np.einsum("ti,ij,tj->t", X, XtX_inv, X)
s2 = (e @ e) / (n - k)
D = (e**2 / (k * s2)) * h / (1 - h)**2
lev = pd.Series(h, index=idx); cook = pd.Series(D, index=idx)
print("top leverage days:"); print(lev.nlargest(5).round(4).to_string())
print(f"h threshold 3k/n = {3*k/n:.4f}   share above: {(h > 3*k/n).mean():.3%}")
print("\ntop Cook's D days:"); print(cook.nlargest(5).round(4).to_string())
print(f"D threshold 4/n = {4/n:.5f}   share above: {(D > 4/n).mean():.3%}")

**Expected pattern.** Top-h days: the calendar's most violent market days
(2020-03, 2018-02 in real data) — leverage is a property of **x only**,
so it flags market-history, not your model's failures. Top-D days: a
*different, shorter* list — high-h days whose residuals were ALSO large
(the relationship broke that day); many top-h days stay off the D list
because the stock cooperated with its β. The quadrant table writes itself
— high h + small e: harmless pinions; low h + big e: ordinary noise; high
h + big e: the regime-breakers worth three lines of your log each.

## E3 — delete-and-report

In [ ]:
drop5 = cook.nlargest(5).index
keep = ~idx.isin(drop5)
g = ols_fit(x[keep], rr["AAPL"].values[keep])
print(f"beta full {f['b'][1]:.4f} (SE {f['se'][1]:.4f})  ->  no-top5 {g['b'][1]:.4f} (SE {g['se'][1]:.4f})")
print(f"shift {(g['b'][1]-f['b'][1]):+.4f} = {(g['b'][1]-f['b'][1])/f['se'][1]:+.2f} SEs   days: {[str(d.date()) for d in drop5]}")

**Expected pattern (real AAPL).** The shift is typically ~0.1–0.4 SE —
present but not story-changing: the mega-cap market model is *not* a
five-days artifact; the crash days mostly sat near a flatter local line
and trimmed the slope slightly. When you see ≥ 1 SE here (you will, on
some names — financials in 2008–09), the honest sentence is "the beta is
conditional: crash-window observations carry it from β_calm to β_full;
day 12's interaction will model exactly that." **The protocol's deliverable
is never the deletion — it is the sentence quantifying whose β you
measured.** Common mistake: deleting and steamrolling on, destination
unchanged — the log entry is the point, the refit was the measurement.

## E4 — manufactured nonlinearity, caught

In [ ]:
rng = np.random.default_rng(31)
xm = rng.normal(0, 0.01, 3000)
ym = 0.7 * xm + 0.0008 * (100 * xm)**2 * (xm < 0) + rng.normal(0, 0.008, 3000)
fm = ols_fit(xm, ym)
em = fm["resid"]

fig, ax = plt.subplots(1, 3, figsize=(12, 3.5))
ax[0].scatter(fm["yhat"], em, s=4, alpha=0.3); ax[0].set_title("e vs fitted")
ax[1].scatter(xm, em, s=4, alpha=0.3); ax[1].set_title("e vs x")
ax[2].scatter(xm, np.abs(em), s=4, alpha=0.3); ax[2].set_title("|e| vs x")
plt.tight_layout(); plt.show()

Xq = np.column_stack([np.ones(len(xm)), xm, xm**2])
bq = np.linalg.solve(Xq.T @ Xq, Xq.T @ ym)
eq = ym - Xq @ bq
print(f"linear fit: beta_x {fm['b'][1]:.3f} | quad fit: beta_x {bq[1]:.3f}, beta_x2 {bq[2]:+.4f}")
print(f"mean|e| linear {np.abs(em).mean():.5f} -> quadratic {np.abs(eq).mean():.5f}")

**Expected pattern.** Panel e-vs-x shows an asymmetric wing — the left
(down-market) side bows upward — exactly the planted crash-squared term;
e-vs-fitted shows the same as a left-end hook. The quadratic fit absorbs
it (β̂_x² positive and significant; mean|e| drops). **The diagnostic
sequence worked end-to-end: panels localized the failure to "negative large
x", specification cured it (a day-12 preview), and no assumption-repair
(White/NW) could have — A1 failures demand respecification, never a
sandwich.**

## E5 — the boundary (exemplar)

**For exclusion:** 2020-03-16 sits at leverage ~10× average with a
residual in the top tail; if the model's mandate is *calm-regime
portfolio construction*, an estimator whose variance the crash dominates
mis-serves the mandate, and trimming to the regime of use is
specification coherence, not cosmetics.

**Against exclusion:** the crash is sampled information from the same
world you'll trade in — deleting it deletes your best estimate of *when
the beta matters most*; the result will be quoted out-of-regime anyway;
and "outlier" here is just "extreme realization", not a data error
(module 05's error taxonomy: this is news, not noise).

**Disposition (exemplar):** keep, winsorize aggressively if the tails
break machinery downstream (day 11), and ALWAYS add the crash-interaction
specification (day 12) so the model can say "β is higher in stress" out
loud instead of smuggling it into one average slope. Report both fits.
Deletions require a data-error justification, and "inconveniently large"
fails that test.